# Projet Python pour la Data Science : Influence des médailles remportées par la France aux Jeux Olympiques sur le nombre le licenciés sportifs en France.

_Autrices : Melissa MIGAN, Camille PEYTHIEUX-TALDIR, Romane PLUQUET_.

## Introduction

# to do

### Sommaire

#TODO
[Introduction](#introduction)

[Données](#données)

[Analyse](#analyse)

[Conclusion](#conclusion)

## I. Création et exploration de la base de données principale

Dans cette partie, l'objectif est d'importer et de travailler les différentes bases de données et de les joindre en une base de données exploitable. Après travail et nettoyage des données brutes, nous utilisons trois bases de données en accès public :

| Nom de la base   | Description                                                                 | Source     | Mode d'extraction |
|-----------------|------------------------------------------------------------------------------|------------|-------------------|
| `data_medailles`  | Nombre de médailles reportées par la France aux Jeux Olympiques par sport et année (2016-2024). | Wikipedia  | Web scraping      |
| `data_licences`   | Nombre de licenciés sportifs en France par fédération, sexe, âge et département (2016-2024).       | Injep (Institut national de la jeunesse et de l'éducation populaire)          | CSV, Parquet      |
| `data_pop`        | Population départementale en France (recensements de 2016 et 2022).          | Insee      | API               |

La dernière (`data_pop`) n'étant utilisée que dans un graphique (dans un but de pondération), elle ne sera pas incluse dans notre base de données principale.

La table data_licences recense les effectif de licenciés par an entre 2016 et 2024. Elle présente également des effectifs par tranches d'âge et par genre. La table data_pop contient les populations départementales recensées en 2016 et en 2022. Elle nous permet de mener une étude des effectifs de licenciés par département, relativement à la population de ces derniers. Finalement, la table data_medailles rassemble les médailles obtenues par la France aux Jeux Olympiques entre 2016 et 2024, c'est-à-dire aux JO de 2016, 2020 (qui ont eu lieu en 2021 à cause du Covid) et de 2024. 

On importe les modules pour traiter les données, et les fonctions utilisées.

In [ ]:
#%pip install -r requirements.txt

# Modules
#import os
import pandas as pd
import numpy as np
import pyarrow as pa

# Fonctions
from data import (
    gel_tableau_medailles,
    nettoyer_base,
    fusionner_bases,
    reorganiser_colonnes,
    normalisation_unicode,
    code_sport,
    code_dep,
    renommer_colonnes, 
    gel_licences,
    tableau_ratios_nr
)

### A. Récupération des données

#### 1. Médailles françaises aux Jeux Olympiques

Nous avons scrappé la page Wikipedia ["_France aux Jeux Olympiques_"](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques) afin d'obtenir les tableaux des médailles (or, argent bronze) obtenues par la France lors des Jeux Olympiques, à l'aide de la fonction `tableau_scraper`.

In [ ]:
a_figer = [["or", "M.C3.A9dailles_d.27or_3"],
           ["argent", "M.C3.A9dailles_d.27argent"],
           ["bronze", "M.C3.A9dailles_de_bronze"]]

for duo in a_figer:
    #gel_tableau_medailles(duo[0], duo[1])

 Nous avons ensuite gelé les bases scrapées dans un souci de reproductibilité, au cas où la page Wikipédia soit modifiée. Nous avons ainsi obtenu trois tables :
- `data_or` : renseignant le nombre de médailles d'or obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_argent` : renseignant le nombre de médailles d'argent obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_bronze` : renseignant le nombre de médailles de bronze obtenues par la France aux Jeux Olympiques, par sport et par année.

In [ ]:
data_or = pd.read_csv(f"data/data_medailles/data_or_jo.csv")
data_argent = pd.read_csv(f"data/data_medailles/data_argent_jo.csv")
data_bronze = pd.read_csv(f"data/data_medailles/data_bronze_jo.csv")

display(data_or.head())
display(data_argent.head())
display(data_bronze.head())

Nous avons ensuite nettoyé ces tables, en enlevant les années et sports ne nous intéressant pas, ainsi que les lignes et colonnes vides ou de total.

In [ ]:
data_or_clean = nettoyer_base(data_or)
data_argent_clean = nettoyer_base(data_argent)
data_bronze_clean = nettoyer_base(data_bronze)

display(data_or_clean.head())
display(data_argent_clean.head())
display(data_bronze_clean.head())

Suite à cela, nous avons joint ces trois tables afin d'obtenir la table `data_medailles_jo`, renseignant le nombre de médailles (toutes couleurs confondues) obtenues par la France aux Jeux Olympiques de 2016, 2020 et 2024. Nous y avons ajouté un code sport, permettant plus tard la jointure avec la table des licenciés sportifs en France.

In [ ]:
data_medailles = fusionner_bases(data_or, data_argent, data_bronze).head()
display(data_medailles.head())

#### 2. Licenciés sportifs en France

Nous avons exploité les données de licenciés en France fournies par l'Injep, l'Institut national de la Jeunesse et de l'Education populaire. Les données se présentent sous forme brute comme un fichier .csv par an. Cependant, ces fichiers étant trop lourds pour être importés directement dans Git, nous optons pour une gestion des données par fichiers au format .parquet. 


##### a. Construction de la base 

Notre but ici est de contruire une base de données au format long. Chaque base disposant déjà d'une colonne année, nous devons alors les concaténer pour obtenir la base au format désiré. Pour que l'opération se déroule correctement, nous réorganisons dans un premier temps les colonnes de chacune des bases, de sorte que chacune ait les mêmes colonnes dans le même ordre.

In [ ]:
data_licences = reorganiser_colonnes()
data_licences = pa.concat_tables(data_licences)

Afin d'éviter les problèmes de sélection de données, car nous disposons de variables dont les modalités sont textuelles, nous normalisons tous les caractères avec la norme unicode. Nous ajoutons ensuite plusieurs variables permettant un niveau d'analyse plus général que celui très fin proposé par la base de données : 
- `code_sport` : catégorise les fédérations selon le sport pratiqué. Nous choisissons de catégoriser en "divers" (`code_sport` : DIV) les fédérations qui pratiquent un sport non-olympique. Ce code est le même que celui ajouté à la table des médailles. 
- `code_dep` : indique le département par son numéro seulement. 

In [ ]:
data_licences = normalisation_unicode(data_licences)
data_licences = code_sport(data_licences)
data_licences = code_dep(data_licences)

Nous renommons ensuite les colonnes pour avoir des noms de variable sans majusucles ni accents. Nous ne gardons que les colonnes qui nous seront utiles pour la suite, à savoir celles concernant le nom de la fédération, l'année de recensement des licences, le sexe des licencié.e.s, les tranches d'âge (age, tranches fines et grandes fines), le nombre de licences annuelles, le code sport et le code département. Finalement, nous gelons la table dans un fichier .parquet pour la réutiliser par la suite telle que construite ici. 

In [ ]:
data_licences = renommer_colonnes(data_licences)
data_licences = data_licences[["federation","annee", "sexe", "age", "tranche_age","grande_tranche_age","licences_annuelles","code_sport","code_dep"]]
#gel_licences(data_licences)
data_licences.sample(5)

#TODO: décrire base? (en v2v est ce que je décris toutes les tranches d'âge?)

##### b. Précautions : comparabilité dans le temps

Nos données de licences proviennent de fichiers distincts pour chaque année recensée. Ces fichiers sont de deux types :
- `semidef` pour les années 2016 à 2018 et 2023 à 2024, qui n'ont pas encore été "géocodées",
- `def` pour les années 2019 à 2022, qui sont "géocodées".

La documentation de ces données affirme que les données sont comparables entre elles dans le temps à condition d'être agrégées par fédération. Cependant, il est précisé que les "données par sexe, et/ou par âge, et/ou par département/région ne doivent pas être
comparées dans le temps directement". Il faut dans ce cas là bien prendre en compte les effectifs non répartis (NR), c'est-à-dire qui n'ont pas pu être classés selon un département, une catégorie d'âge ou de genre, afin d'obtenir des résultats comparables dans le temps. 

Afin d'avoir une idée de l'ampleur de la non répartition géographique des effectifs de licences dans les données, nous calculons les ratios d'effectifs de licences géographiquement non répartis sur la totalité des effectifs de licences par an, ainsi que dans la base regroupant toutes les années. 

In [ ]:
display(tableau_ratios_nr(data_licences, "code_dep"))

On constate alors que la proportion de licences non géographiquement réparties est bien plus important pour les premiers fichiers `semidef`, c'est-à-dire de 2016 à 2018, alors que par la suite, cette proportion n'excède pas les 1%. 

In [ ]:
display(tableau_ratios_nr(data_licences, "sexe"))
display(tableau_ratios_nr(data_licences, "age"))
display(tableau_ratios_nr(data_licences, "tranche_age"))
display(tableau_ratios_nr(data_licences, "grande_tranche_age"))

#### 3. Population départementale en France

### B. Jointure des tables

Nous avons joint les tables médailles et licences pour pouvoir étudier en détail l'effet de remporter des médailles aux Jeux Olympiques sur l'évolution du nombre de licenciés sportifs. Nous avons joint par la gauche en utilisant la clé `code_sport` pour ne pas démultiplier le nombre de lignes dans notre DataFrame : nous avons un unique code sport par ligne dans norte table médailles.

In [ ]:
df_lic = pd.read_parquet("data/data_licences/data_licences.parquet")

data_complet = pd.merge(df_lic, data_medailles, how='left', on="code_sport")
data_complet.head()

to do

In [ ]:
#data_complet.to_parquet("data_complet.parquet")


OUTPUT_DIR.mkdir(exist_ok=True)

musees.to_csv(OUTPUT_DIR / "musees.csv", index=False)
frequentation_annuelle.to_csv(OUTPUT_DIR / "frequentation_annuelle.csv", index=False)
freq_excel_long.to_csv(OUTPUT_DIR / "frequentation_excel_long.csv", index=False)
df_modele_clean.to_csv(OUTPUT_DIR / "df_modele_musees.csv", index=False)

print("Fichiers exportés dans :", OUTPUT_DIR.resolve())


# Contrôles qualité et tests : todo